# Mercedes constant-speed validation: speed-blind inference
Reference speeds and text labels are not used by the detector or geometry model.

In [ ]:
import subprocess,sys
subprocess.check_call([sys.executable,'-m','pip','install','-q','ultralytics==8.4.175','lap==0.5.13'])

In [ ]:
from pathlib import Path
source='"""Speed-blind detection and cuboid extraction for constant-speed validation clips."""\nfrom pathlib import Path\nimport argparse\nimport hashlib\nimport json\nimport time\nimport urllib.request\n\nimport cv2\nimport numpy as np\nfrom scipy.optimize import linear_sum_assignment\nfrom ultralytics import YOLO\n\nMODEL_REPO=\'mehmetkeremturkcan/UrbanOmniDetect\'\nMODEL_REVISION=\'2701a2eec9a73afb5ffd852740d125dfcd065ce5\'\nMODEL_FILE=\'checkpoints/urbanomnidetect_yolo11x-p2_1920.pt\'\nMODEL_SHA=\'9aa55280a5ef1c9d89c0acb5baeb974b90fbdf3c1deead1413b24d7af249226d\'\n\n\ndef overlaps(a,b):\n    lo=np.maximum(a[:,None,:2],b[None,:,:2]);hi=np.minimum(a[:,None,2:],b[None,:,2:])\n    intersection=np.prod(np.maximum(0,hi-lo),axis=2)\n    return intersection/(np.prod(a[:,2:]-a[:,:2],axis=1)[:,None]+\n                         np.prod(b[:,2:]-b[:,:2],axis=1)[None,:]-intersection+1e-9)\n\n\ndef infer(folder,output,device=\'0\'):\n    output=Path(output);output.mkdir(parents=True,exist_ok=True)\n    model_dir=output.parent/\'models\';model_dir.mkdir(exist_ok=True)\n    weights=model_dir/\'urban_geometry.pt\'\n    if not weights.exists():\n        urllib.request.urlretrieve(f\'https://huggingface.co/{MODEL_REPO}/resolve/{MODEL_REVISION}/{MODEL_FILE}\',weights)\n    assert hashlib.sha256(weights.read_bytes()).hexdigest()==MODEL_SHA\n    geometry_model=YOLO(str(weights))\n    for video in sorted(Path(folder).rglob(\'*.MP4\')):\n        target=output/video.stem;target.mkdir(exist_ok=True)\n        tracker=YOLO(str(model_dir/\'yolo11s.pt\'))\n        cap=cv2.VideoCapture(str(video));assert cap.isOpened()\n        width=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH));height=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))\n        fps=float(cap.get(cv2.CAP_PROP_FPS));total=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))\n        observations=[];tracks=[];times=[];start=time.monotonic();index=0\n        while True:\n            ok,frame=cap.read()\n            if not ok:break\n            pts=float(cap.get(cv2.CAP_PROP_POS_MSEC))/1000\n            timestamp=pts if abs(pts-index/fps)<.1 else index/fps\n            times.append(timestamp)\n            if index==0:cv2.imwrite(str(target/\'reference.jpg\'),frame)\n            detected=tracker.track(frame,persist=True,tracker=\'bytetrack.yaml\',device=device,\n                                   imgsz=1280,classes=[2,5,7],conf=.15,verbose=False)[0]\n            boxes=detected.boxes.xyxy.cpu().numpy()\n            ids=detected.boxes.id.cpu().numpy().astype(int) if detected.boxes.id is not None else np.full(len(boxes),-1)\n            scores=detected.boxes.conf.cpu().numpy()\n            for box,tid,score in zip(boxes,ids,scores):\n                tracks.append({\'frame_index\':index,\'time_s\':timestamp,\'track_id\':int(tid),\n                               \'box\':box.tolist(),\'confidence\':float(score)})\n            selected=[i for i,box in enumerate(boxes) if box[3]-box[1]>=45 and ids[i]>=0]\n            if selected:\n                result=geometry_model.predict(frame,device=device,imgsz=1920,classes=[0],conf=.15,verbose=False)[0]\n                if result.keypoints is not None and len(result.boxes):\n                    gb=result.boxes.xyxy.cpu().numpy();gs=result.boxes.conf.cpu().numpy()\n                    kp=result.keypoints.data.cpu().numpy()\n                    overlap=overlaps(boxes[selected],gb);ii,jj=linear_sum_assignment(1-overlap)\n                    for i,j in zip(ii,jj):\n                        if overlap[i,j]<.25:continue\n                        k=selected[i]\n                        observations.append({\'frame_index\':index,\'time_s\':timestamp,\'track_id\':int(ids[k]),\n                            \'baseline_confidence\':float(scores[k]),\'geometry_confidence\':float(gs[j]),\n                            \'association_iou\':float(overlap[i,j]),\'baseline_box\':boxes[k].tolist(),\n                            \'geometry_box\':gb[j].tolist(),\'keypoints\':kp[j].tolist()})\n                    if len(observations)%20<2:\n                        cv2.imwrite(str(target/f\'geometry_{index:04d}.jpg\'),result.plot())\n            index+=1\n        cap.release()\n        assert index==total\n        (target/\'observations.json\').write_text(json.dumps(observations),encoding=\'utf-8\')\n        (target/\'tracks.json\').write_text(json.dumps(tracks),encoding=\'utf-8\')\n        if observations:\n            points=np.array([r[\'keypoints\'] for r in observations])\n            vote=points[:,:4,1].mean(axis=1)>points[:,4:,1].mean(axis=1)\n            ground_indices=list(range(4)) if vote.mean()>=.5 else list(range(4,8))\n            agreement=float(max(vote.mean(),1-vote.mean()))\n        else:ground_indices=[];agreement=0.0\n        report={\'source_video\':video.name,\'width\':width,\'height\':height,\'fps\':fps,\'frames\':total,\n                \'duration_s\':total/fps,\'timestamps\':times,\'model_sha256\':MODEL_SHA,\n                \'model_revision\':MODEL_REVISION,\'model_filename\':MODEL_FILE,\'ground_indices\':ground_indices,\n                \'ground_order_vote_agreement\':agreement,\'runtime_s\':time.monotonic()-start,\n                \'observations\':len(observations),\'reference_speeds_used_by_inference\':False}\n        (target/\'geometry_run_report.json\').write_text(json.dumps(report,indent=2),encoding=\'utf-8\')\n        print(video.name,\'frames\',index,\'geometry\',len(observations),\'seconds\',round(report[\'runtime_s\'],1),flush=True)\n\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--input\',required=True);parser.add_argument(\'--output\',required=True)\n    parser.add_argument(\'--device\',default=\'0\')\n    args=parser.parse_args();infer(args.input,args.output,args.device)\n'
Path('/kaggle/working/validation_inference.py').write_text(source)
subprocess.check_call([sys.executable,'/kaggle/working/validation_inference.py','--input','/kaggle/input','--output','/kaggle/working/validation_inference','--device','0'])